<b>Multi-Commodity Flow Problem</b>

In [1]:
# data
Nodes = range(1,9)
Capacity = { (1,3) : 300, (1,4) : 300, (2,3) : 300, (2,4) : 300, (3,4) : 300, (3,5) : 300, (3,6) : 300, (4,3) : 300, (4,5) : 300, (4,7) : 300, (5,3) : 300, (5,4) : 300, (5,6) : 300, (5,7) : 300, (5,8) : 300, (6,3) : 300, (6,5) : 300, (6,8) : 300, (7,4) : 300, (7,5) : 300, (7,8) : 300, (8,5) : 300, (8,6) : 300, (8,7) : 300 }
Cost = { (1,3) : 20, (1,4) : 35, (2,3) : 45, (2,4) : 30, (3,4) : 25, (3,5) : 25, (3,6) : 34, (4,3) : 25, (4,5) : 26, (4,7) : 31, (5,3) : 25, (5,4) : 26, (5,6) : 29, (5,7) : 26, (5,8) : 28, (6,3) : 34, (6,5) : 29, (6,8) : 37, (7,4) : 31, (7,5) : 26, (7,8) : 40, (8,5) : 28, (8,6) : 37, (8,7) : 40 }
Arcs = { (i,j) for (i,j) in Cost}

Commodities = {'C1', 'C2', 'C3', 'C4' }
Origin = { 'C1' : 1, 'C2' : 1, 'C3' : 2, 'C4' : 2 }
Destination = { 'C1' : 7, 'C2' : 6, 'C3' : 7, 'C4' : 8 }
Volume = { 'C1' : 200, 'C2' : 300, 'C3' : 150, 'C4' : 240 }

In [2]:
# define supply/demand for each node and each commodity
Supply = {}
for i in Nodes:
    for k in Commodities:
        if i==Origin[k]:
            Supply[i,k] = -Volume[k]
        elif i==Destination[k]:
            Supply[i,k] = Volume[k]
        else:
            Supply[i,k] = 0
       

In [3]:
Supply

{(1, 'C1'): -200,
 (1, 'C2'): -300,
 (1, 'C4'): 0,
 (1, 'C3'): 0,
 (2, 'C1'): 0,
 (2, 'C2'): 0,
 (2, 'C4'): -240,
 (2, 'C3'): -150,
 (3, 'C1'): 0,
 (3, 'C2'): 0,
 (3, 'C4'): 0,
 (3, 'C3'): 0,
 (4, 'C1'): 0,
 (4, 'C2'): 0,
 (4, 'C4'): 0,
 (4, 'C3'): 0,
 (5, 'C1'): 0,
 (5, 'C2'): 0,
 (5, 'C4'): 0,
 (5, 'C3'): 0,
 (6, 'C1'): 0,
 (6, 'C2'): 300,
 (6, 'C4'): 0,
 (6, 'C3'): 0,
 (7, 'C1'): 200,
 (7, 'C2'): 0,
 (7, 'C4'): 0,
 (7, 'C3'): 150,
 (8, 'C1'): 0,
 (8, 'C2'): 0,
 (8, 'C4'): 240,
 (8, 'C3'): 0}

In [4]:
# define a set of arc/commodity pairs (useful for variables)
ArcCommodities = { (i,j,k) for (i,j) in Arcs for k in Commodities }

In [5]:
from docplex.mp.model import Model
mdl = Model()

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [6]:
# define variables
# note that we cannot directly impose upper bounds via 'Capacity[i,j]' as this parameter 
# cannot be mapped unambiguously onto the tuple (i,j,k) for which the variable is defined.
shipped = mdl.continuous_var_dict(ArcCommodities, lb=0, name='shipped')

In [6]:
# objective
mdl.minimize(mdl.sum(Cost[i,j]*shipped[i,j,k] for (i,j,k) in shipped))

In [7]:
# flow balance constraints for each commodity and for each node
for k in Commodities:
    for j in Nodes:
        inflow = sum(shipped[i,j,k] for i in Nodes if (i,j) in Arcs)
        outflow = sum(shipped[j,i,k] for i in Nodes if (j,i) in Arcs)
        mdl.add_constraint(inflow - outflow == Supply[j,k])

In [8]:
# define arc capacity constraints (over all commodities)
for (i,j) in Arcs:
    mdl.add_constraint(mdl.sum(shipped[i,j,k] for k in Commodities) <= Capacity[i,j])

In [9]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.016,status='optimal')

In [10]:
mdl.print_solution()

objective: 61020.000
  shipped_2_4_C3=150.000
  shipped_5_8_C4=240.000
  shipped_3_6_C2=300.000
  shipped_4_7_C1=150.000
  shipped_5_7_C1=50.000
  shipped_4_7_C3=150.000
  shipped_4_5_C1=50.000
  shipped_3_5_C4=90.000
  shipped_1_3_C2=300.000
  shipped_2_4_C4=150.000
  shipped_4_5_C4=150.000
  shipped_1_4_C1=200.000
  shipped_2_3_C4=90.000
